# Lesson 28: Prompting techniques I — shots, chain of thought, self-consistency, role and context 🎯
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Define **prompt engineering** and why it works
2. **Measure** the difference between **zero-shot, one-shot and few-shot** prompting
3. Use **chain-of-thought (CoT)** and **self-consistency** to improve reasoning, and measure it
4. Use **role** and **contextual** prompting, and know their limits

**⏱️ Timing:** shots 15' | CoT 15' | self-consistency 15' | role & context 15'

> ⚙️ **GPU recommended:** Runtime → Change runtime type → **T4 GPU** (the local LLM runs much faster).

## 🧠 What is prompt engineering?
**Prompt engineering** = designing the input to an LLM so it reliably gives the output you need — **without changing the model's weights**.

Why does wording matter so much? Remember Lesson 5: an LLM predicts the most likely continuation. A prompt sets up the **situation** the model is continuing. Better situation → better continuation.

In Lesson 12 we learned the basics (role, task, context, format). Today we act like scientists: **change one thing, measure the result.**

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install openai
from openai import OpenAI
import re, json
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def ask(prompt, system=None, temperature=0.0, max_tokens=400):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

print(ask("Say hello in five words."))

## Part 1: Zero-shot, one-shot, few-shot 🎯
- **Zero-shot**: just the instruction, no examples
- **One-shot**: one worked example
- **Few-shot**: several examples (GPT-3 made this famous in 2020)

**Task:** sort messages to a school office into `ATTENDANCE`, `BULLYING`, `LOST_PROPERTY`, `IT_HELP` or `OTHER`. We'll score each method on 10 test messages.

In [ ]:
tests = [
    ("My son has a fever and won't be at school today.", "ATTENDANCE"),
    ("Someone keeps posting mean things about me in the class group chat.", "BULLYING"),
    ("I left my green drink bottle in the gym on Tuesday.", "LOST_PROPERTY"),
    ("I can't log in to the school laptop, it says my password expired.", "IT_HELP"),
    ("What time does the school disco finish?", "OTHER"),
    ("Mia will be leaving early at 1pm for a dentist appointment.", "ATTENDANCE"),
    ("Has anyone handed in a black calculator?", "LOST_PROPERTY"),
    ("Some older boys keep pushing my brother at the bus stop.", "BULLYING"),
    ("The classroom printer is showing error 49.", "IT_HELP"),
    ("Can parents volunteer at the sausage sizzle?", "OTHER"),
]
LABELS = ["ATTENDANCE", "BULLYING", "LOST_PROPERTY", "IT_HELP", "OTHER"]
instruction = f"Classify the message into exactly one category: {', '.join(LABELS)}. Reply with the category only."

examples = [("Tom is sick and will stay home tomorrow.", "ATTENDANCE"),
            ("My headphones went missing after PE.", "LOST_PROPERTY"),
            ("The Wi-Fi in the library isn't working.", "IT_HELP"),
            ("A girl in my class says nobody should sit with me.", "BULLYING"),
            ("When is the next parent-teacher evening?", "OTHER")]

def build_prompt(message, n_examples):
    shots = "".join(f"Message: {m}\nCategory: {c}\n\n" for m, c in examples[:n_examples])
    return f"{instruction}\n\n{shots}Message: {message}\nCategory:"

def score(n_examples, show=False):
    correct = 0
    for msg, label in tests:
        answer = ask(build_prompt(msg, n_examples), max_tokens=10).upper()
        ok = label in answer and all(l not in answer for l in LABELS if l != label)
        correct += ok
        if show and not ok: print(f"   ❌ {msg[:45]:45} → {answer!r} (should be {label})")
    return correct / len(tests)

for name, n in [("zero-shot", 0), ("one-shot", 1), ("few-shot (5)", 5)]:
    print(f"{name:13} accuracy: {score(n, show=True):.0%}")

In [ ]:
print("What the few-shot prompt actually looks like:\n")
print(build_prompt("I lost my jacket.", 5))

### 🧩 Exercise 1
1. Which mistakes did zero-shot make — wrong **category**, or wrong **format** (extra words)? Few-shot examples teach both.
2. Make the examples **bad**: give 5 examples that are all `ATTENDANCE`. What happens? (Models copy patterns in examples — including bias!)
3. Add 5 tricky test messages of your own.

## Part 2: Chain-of-thought (CoT) 🔗
Ask the model to **show its working** before the answer. Writing the steps gives the model "space to think": each step becomes context for the next prediction.

In [ ]:
problems = [
    ("A class has 28 students. 3/4 of them like pizza. How many students do NOT like pizza?", 7),
    ("Ana buys 3 notebooks at $4 each and a pen for $2. She pays with a $20 note. How much change does she get?", 6),
    ("A bus leaves at 8:45 and the trip takes 50 minutes. At what minute past 9 does it arrive?", 35),
    ("There are 5 shelves with 12 books each. 17 books are borrowed. How many books are left?", 43),
    ("Liam is twice as old as his sister. In 3 years their ages will add up to 30. How old is Liam now?", 16),
    ("A rectangle is 9 cm long and has a perimeter of 30 cm. What is its width in cm?", 6),
    ("Tickets cost $7 for students and $12 for adults. 4 students and 2 adults go. What is the total cost in dollars?", 52),
    ("A tank holds 120 litres. It is 25% full. How many more litres are needed to fill it?", 90),
]

def last_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(nums[-1]) if nums else None

def run(style, temperature=0.0):
    correct = 0
    for q, ans in problems:
        if style == "direct":
            out = ask(q + " Reply with only the final number.", temperature=temperature, max_tokens=20)
        else:
            out = ask(q + " Let's think step by step, then write the final answer as 'Answer: <number>'.", temperature=temperature)
        correct += last_number(out) == ans
    return correct / len(problems)

print(f"Direct answer:     {run('direct'):.0%}")
print(f"Chain of thought:  {run('cot'):.0%}")

In [ ]:
print(ask(problems[4][0] + " Let's think step by step, then write the final answer as 'Answer: <number>'."))

### 🧩 Exercise 2
1. Read the CoT working for a problem it got wrong. Where exactly did the reasoning go wrong?
2. CoT costs more tokens (and time). For which kinds of questions is it worth it? Which not? (*"What's the capital of France?"*)

> 💡 Newer "reasoning models" are trained to think step by step on their own before answering, so you often don't need to ask. Small models still benefit a lot.

## Part 3: Self-consistency — ask several times, take a vote 🗳️
With **temperature > 0**, each CoT run takes a slightly different path. If most paths reach the same answer, it's more likely right. (Like asking 5 classmates to solve a problem independently and trusting the majority.)

In [ ]:
from collections import Counter

def self_consistent(q, n=5):
    votes = []
    for _ in range(n):
        out = ask(q + " Let's think step by step, then write the final answer as 'Answer: <number>'.", temperature=0.8)
        votes.append(last_number(out))
    winner, count = Counter(votes).most_common(1)[0]
    return winner, votes

correct = 0
for q, ans in problems:
    winner, votes = self_consistent(q)
    correct += winner == ans
    print(f"{'✅' if winner == ans else '❌'} votes {votes} → {winner} (true {ans})")
print(f"\nSelf-consistency (5 votes): {correct / len(problems):.0%}")

### 🧩 Exercise 3
1. Compare all three accuracies. Was self-consistency worth **5×** the cost?
2. When the votes are split (e.g. 2–2–1), what does that tell you about the model's confidence? How could an app use that?

## Part 4: Role prompting and contextual prompting 🎭📎
- **Role prompting**: give the model a persona ("You are a primary school teacher…")
- **Contextual prompting**: give it the **background facts** it needs

In [ ]:
question = "Explain what a black hole is."
for role in [None, "You are a primary school teacher talking to 7-year-olds.",
             "You are an astrophysics professor writing for university students.",
             "You are a sports commentator."]:
    print(f"== role: {role}\n{ask(question, system=role, max_tokens=120)}\n")

In [ ]:
q = "Can I bring my phone on the camp, and what time do we leave?"
context = """Year 9 Camp information:
- Camp is at Lake Rotoiti, 14–16 March.
- The bus leaves the school gym at 7:30am sharp on the 14th.
- Phones must be handed to teachers on the bus and are returned on the last day.
- Bring: sleeping bag, warm jacket, togs, water bottle."""
print("WITHOUT context:\n", ask(q, max_tokens=120), "\n")
print("WITH context:\n", ask(f"Use only this information to answer.\n\n{context}\n\nQuestion: {q}", max_tokens=120))

### 🧩 Exercise 4
1. Without context, did the model **make up** camp rules? (That's hallucination — Lesson 14.)
2. Does saying "You are a world-class physicist" make the model **know more** physics? Test it on a hard question with and without the role. (Research suggests roles change **style and audience** much more than **accuracy**.)
3. Contextual prompting is the seed of **RAG** — automatically finding the right context to paste in (Lessons 29 and 32).

## 📌 Summary
| Technique | What you do | Best for |
|---|---|---|
| Zero-shot | Instruction only | Easy, common tasks |
| One/few-shot | Add worked examples | Exact formats, custom categories |
| Chain of thought | "Think step by step" | Maths, logic, multi-step problems |
| Self-consistency | Several CoT samples + vote | When accuracy matters more than cost |
| Role | Give a persona | Tone, audience, style |
| Contextual | Paste the facts in | Questions about **your** information |